# Bootstrap Validation — Random Forest, Optuna 30 trials

Identical to nb26 pipeline but restricted to **Random Forest only** and **Optuna 30 trials** (matching nb21 exactly).

**Purpose:** Fair comparison against nb29 (fixed-param RF, B=100). If bootstrap mean here is notably higher than nb29's RF mean (0.760), tuning helps. If similar, the signal is in the data and fixed params suffice.

**Pipeline per iteration:** Optuna 30 trials → CV SHAP → genre consolidation → forward selection → Optuna re-tune 30 trials.  
**No centrality ablation** (consistent with nb26).  
Results saved to `bootstrap_rf_30t_results.pkl` after every iteration.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import matplotlib.gridspec as gridspec

import shap
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split, StratifiedKFold

In [2]:
B            = 25
N_TRIALS     = 30
LAM          = 0.3
N_SPLITS     = 5
MIN_N        = 5
RANDOM_STATE = 42

# Reference lines
NB21_RF_AUC  = 0.7671
NB29_RF_MEAN = 0.7602
NB26_RF_MEAN = 0.7447

df = pd.read_csv('../df_artists_final.csv', index_col=0).reset_index()
X  = df.drop(columns=['top_20_hitmaker'])
y  = df['top_20_hitmaker']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Class balance (train): {y_train.mean():.3f} hitmaker')

Train: (607, 26)  Test: (152, 26)
Class balance (train): 0.432 hitmaker


In [4]:
def build_rf(params):
    return RandomForestClassifier(
        n_estimators=params['n_estimators'],
        max_depth=params.get('max_depth'),
        min_samples_leaf=params.get('min_samples_leaf', 1),
        max_features=params.get('max_features', 'sqrt'),
        class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1,
    )


def make_optuna_objective(X, y, lam, skf, seed):
    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
            'max_depth':        trial.suggest_int('max_depth', 2, 15),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
            'max_features':     trial.suggest_categorical('max_features',
                                    ['sqrt', 'log2', 0.3, 0.5, 0.7]),
        }
        fold_val, fold_tr = [], []
        for tr_idx, val_idx in skf.split(X, y):
            X_f, X_v = X.iloc[tr_idx], X.iloc[val_idx]
            y_f, y_v = y.iloc[tr_idx], y.iloc[val_idx]
            m = build_rf(params)
            try:
                m.fit(X_f, y_f)
                fold_val.append(roc_auc_score(y_v, m.predict_proba(X_v)[:, 1]))
                fold_tr.append(roc_auc_score(y_f,  m.predict_proba(X_f)[:, 1]))
            except Exception:
                fold_val.append(np.nan); fold_tr.append(np.nan)
        val_auc = np.nanmean(fold_val)
        return val_auc - lam * (np.nanmean(fold_tr) - val_auc)
    return objective


def compute_cv_importance(X_tr, y_tr, params, skf):
    fold_imps = []
    for tr_idx, val_idx in skf.split(X_tr, y_tr):
        X_f, X_v = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
        y_f, y_v = y_tr.iloc[tr_idx], y_tr.iloc[val_idx]
        m = build_rf(params)
        try:
            m.fit(X_f, y_f)
            try:
                explainer = shap.TreeExplainer(m)
                sv = explainer.shap_values(X_v)
                if isinstance(sv, list): sv = sv[1]
                elif sv.ndim == 3:       sv = sv[:, :, 1]
                fold_imps.append(np.abs(sv).mean(axis=0))
            except Exception:
                perm = permutation_importance(m, X_v, y_v, n_repeats=5,
                           random_state=RANDOM_STATE, scoring='roc_auc')
                fold_imps.append(perm.importances_mean)
        except Exception:
            pass
    if not fold_imps:
        return pd.DataFrame({'Feature': X_tr.columns,
                             'Importance': np.zeros(X_tr.shape[1])})
    mean_imp = np.mean(fold_imps, axis=0)
    return (pd.DataFrame({'Feature': X_tr.columns, 'Importance': mean_imp})
              .sort_values('Importance', ascending=False).reset_index(drop=True))


def genre_consolidate(X_tr, X_te, perm_df):
    all_genre_cols = [c for c in X_tr.columns if c.startswith('artist_genre_')]
    genre_imp  = perm_df.set_index('Feature')['Importance']
    genre_vals = genre_imp[[c for c in all_genre_cols if c in genre_imp.index]]
    high = [c for c in all_genre_cols if genre_imp.get(c, 0) > genre_vals.mean()]
    low  = [c for c in all_genre_cols if c not in high]
    X_tr_c = X_tr.drop(columns=low).copy()
    X_te_c = X_te.drop(columns=low).copy()
    if low:
        X_tr_c['artist_genre_other'] = (X_tr[low].sum(axis=1) > 0).astype(int)
        X_te_c['artist_genre_other'] = (X_te[low].sum(axis=1) > 0).astype(int)
    return X_tr_c, X_te_c


def forward_select(X_tr, y_tr, params, skf):
    perm_df = compute_cv_importance(X_tr, y_tr, params, skf)
    feature_order = [f for f in perm_df['Feature'] if f in X_tr.columns]
    if 'artist_genre_other' in X_tr.columns and 'artist_genre_other' not in feature_order:
        feature_order.append('artist_genre_other')
    sel_results = []
    for n in range(min(3, len(feature_order)), len(feature_order) + 1):
        feats = feature_order[:n]
        fold_val, fold_tr = [], []
        for tr_idx, val_idx in skf.split(X_tr[feats], y_tr):
            X_f, X_v = X_tr[feats].iloc[tr_idx], X_tr[feats].iloc[val_idx]
            y_f, y_v = y_tr.iloc[tr_idx], y_tr.iloc[val_idx]
            m = build_rf(params)
            try:
                m.fit(X_f, y_f)
                fold_val.append(roc_auc_score(y_v, m.predict_proba(X_v)[:, 1]))
                fold_tr.append(roc_auc_score(y_f, m.predict_proba(X_f)[:, 1]))
            except Exception:
                fold_val.append(np.nan); fold_tr.append(np.nan)
        val_auc = np.nanmean(fold_val)
        sel_results.append({'n_features': n, 'CV AUC': val_auc,
                            'Overfit Gap': np.nanmean(fold_tr) - val_auc})
    return feature_order, pd.DataFrame(sel_results).set_index('n_features')


def run_pipeline(X_tr, y_tr, X_te, y_te, seed):
    skf_inner = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    imp      = SimpleImputer(strategy='median')
    X_tr_imp = pd.DataFrame(imp.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_te_imp = pd.DataFrame(imp.transform(X_te),     columns=X_te.columns, index=X_te.index)

    s1 = optuna.create_study(direction='maximize', sampler=TPESampler(seed=seed))
    s1.optimize(make_optuna_objective(X_tr_imp, y_tr, LAM, skf_inner, seed),
                n_trials=N_TRIALS, show_progress_bar=False)
    params_full = s1.best_params

    perm_df = compute_cv_importance(X_tr_imp, y_tr, params_full, skf_inner)
    X_tr_c, X_te_c = genre_consolidate(X_tr_imp, X_te_imp, perm_df)

    feature_order, df_sel = forward_select(X_tr_c, y_tr, params_full, skf_inner)
    n_peak = max(int(df_sel['CV AUC'].idxmax()), MIN_N)
    feats  = feature_order[:n_peak]

    s2 = optuna.create_study(direction='maximize', sampler=TPESampler(seed=seed))
    s2.optimize(make_optuna_objective(X_tr_c[feats], y_tr, LAM, skf_inner, seed),
                n_trials=N_TRIALS, show_progress_bar=False)
    params_final = s2.best_params

    model = build_rf(params_final)
    model.fit(X_tr_c[feats], y_tr)
    y_proba = model.predict_proba(X_te_c[feats])[:, 1]

    return {
        'test_auc':   roc_auc_score(y_te, y_proba),
        'brier':      brier_score_loss(y_te, y_proba),
        'n_features': n_peak,
    }

## Bootstrap loop

B=25 independent pipeline iterations. Saved to `bootstrap_rf_30t_results.pkl` after every iteration.

In [ ]:
results = {'RF': {'test_auc': [], 'brier': [], 'n_features': []},
           'Baseline': {'test_auc': [], 'brier': []}}

rng = np.random.default_rng(RANDOM_STATE)

for b in range(B):
    seed     = RANDOM_STATE + b
    boot_idx = rng.choice(len(X_train), size=len(X_train), replace=True)
    X_tr_b   = X_train.iloc[boot_idx].reset_index(drop=True)
    y_tr_b   = y_train.iloc[boot_idx].reset_index(drop=True)

    print(f'[{b+1:02d}/{B}] RF... ', end='', flush=True)
    try:
        res = run_pipeline(X_tr_b, y_tr_b, X_test, y_test, seed)
        results['RF']['test_auc'].append(res['test_auc'])
        results['RF']['brier'].append(res['brier'])
        results['RF']['n_features'].append(res['n_features'])
        print(f'AUC={res["test_auc"]:.3f} n={res["n_features"]} | ', end='', flush=True)
    except Exception as e:
        print(f'ERROR({e}) | ', end='', flush=True)
        for k in ('test_auc', 'brier', 'n_features'):
            results['RF'][k].append(np.nan)

    dummy = DummyClassifier(strategy='stratified', random_state=seed)
    dummy.fit(X_tr_b, y_tr_b)
    dummy_proba = dummy.predict_proba(X_test)[:, 1]
    results['Baseline']['test_auc'].append(roc_auc_score(y_test, dummy_proba))
    results['Baseline']['brier'].append(brier_score_loss(y_test, dummy_proba))
    print(f'Baseline={results["Baseline"]["test_auc"][-1]:.3f}')

    with open('bootstrap_rf_30t_results.pkl', 'wb') as f:
        pickle.dump(results, f)

print('\nDone. Results saved to bootstrap_rf_30t_results.pkl')

[01/25] RF... AUC=0.728 n=11 | Baseline=0.492
[02/25] RF... AUC=0.744 n=15 | Baseline=0.506
[03/25] RF... AUC=0.748 n=8 | Baseline=0.501
[04/25] RF... AUC=0.722 n=12 | Baseline=0.511
[05/25] RF... AUC=0.735 n=12 | Baseline=0.502
[06/25] RF... AUC=0.770 n=10 | Baseline=0.522
[07/25] RF... AUC=0.732 n=9 | Baseline=0.463
[08/25] RF... AUC=0.664 n=7 | Baseline=0.519
[09/25] RF... AUC=0.733 n=12 | Baseline=0.493
[10/25] RF... AUC=0.719 n=7 | Baseline=0.543
[11/25] RF... AUC=0.712 n=13 | Baseline=0.422
[12/25] RF... AUC=0.749 n=11 | Baseline=0.435
[13/25] RF... AUC=0.778 n=11 | Baseline=0.505
[14/25] RF... AUC=0.740 n=15 | Baseline=0.436
[15/25] RF... AUC=0.698 n=9 | Baseline=0.524
[16/25] RF... AUC=0.761 n=15 | Baseline=0.603
[17/25] RF... AUC=0.734 n=13 | Baseline=0.509
[18/25] RF... AUC=0.745 n=8 | Baseline=0.510
[19/25] RF... AUC=0.738 n=13 | Baseline=0.478
[20/25] RF... AUC=0.701 n=8 | Baseline=0.599
[21/25] RF... AUC=0.749 n=12 | Baseline=0.528
[22/25] RF... AUC=0.732 n=11 | Baseline=0

## Results

In [ ]:
with open('bootstrap_rf_30t_results.pkl', 'rb') as f:
    results = pickle.load(f)

metric = 'test_auc'
xlim   = (0.40, 0.90)
x_grid = np.linspace(*xlim, 300)

rf_vals       = [v for v in results['RF'][metric] if not np.isnan(v)]
baseline_vals = [v for v in results['Baseline'][metric] if not np.isnan(v)]
baseline_mean = np.mean(baseline_vals)

mu    = np.mean(rf_vals)
std   = np.std(rf_vals)
p5    = np.percentile(rf_vals, 5)
p95   = np.percentile(rf_vals, 95)

# Trimmed mean: drop values more than 2 std below mean
rf_trimmed = [v for v in rf_vals if v >= mu - 2 * std]
mu_trim    = np.mean(rf_trimmed)
n_dropped  = len(rf_vals) - len(rf_trimmed)

kde   = gaussian_kde(rf_vals, bw_method='scott')
y_kde = kde(x_grid)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(rf_vals, bins=10, color='#55A868', alpha=0.35, density=True)
ax.plot(x_grid, y_kde, color='#55A868', lw=2.0)

mask = (x_grid >= p5) & (x_grid <= p95)
ax.fill_between(x_grid[mask], y_kde[mask], alpha=0.30, color='#55A868',
                label=f'90% CI [{p5:.3f}, {p95:.3f}]')

ax.axvline(mu,           color='#55A868', lw=2.0, linestyle='--', label=f'mean={mu:.3f}  (trimmed={mu_trim:.3f}, dropped {n_dropped})')
ax.axvline(mu_trim,      color='#2d7a4f', lw=1.5, linestyle='-.',  alpha=0.8)
ax.axvline(NB21_RF_AUC,  color='black',   lw=1.5, linestyle=':',  label=f'nb21 single-run={NB21_RF_AUC:.3f}')
ax.axvline(NB29_RF_MEAN, color='orange',  lw=1.5, linestyle='--', label=f'nb29 fixed-param mean={NB29_RF_MEAN:.3f}')
ax.axvline(NB26_RF_MEAN, color='grey',    lw=1.5, linestyle='--', label=f'nb26 Optuna-10 mean={NB26_RF_MEAN:.3f}')
ax.axvline(baseline_mean,color='#888888', lw=1.0, linestyle=':',  label=f'Baseline mean={baseline_mean:.3f}')

ax.set_xlabel('Test AUC', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.set_title(f'Random Forest — Bootstrap Validation (B={B}, Optuna {N_TRIALS} trials)\n'
             f'Full pipeline: Optuna → SHAP → genre consolidation → forward selection → re-tune',
             fontsize=12, fontweight='bold')
ax.set_xlim(xlim)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('bootstrap_rf_30t_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nRaw mean    : {mu:.4f}  (B={len(rf_vals)})')
print(f'Trimmed mean: {mu_trim:.4f}  (dropped {n_dropped} outlier(s) below {mu - 2*std:.3f})')


In [ ]:
rf_vals       = [v for v in results['RF']['test_auc']      if not np.isnan(v)]
baseline_vals = [v for v in results['Baseline']['test_auc'] if not np.isnan(v)]
nfeats        = [v for v in results['RF']['n_features']     if not np.isnan(v)]

mu   = np.mean(rf_vals)
std  = np.std(rf_vals)
p5   = np.percentile(rf_vals, 5)
p95  = np.percentile(rf_vals, 95)

print(f'Random Forest — Optuna {N_TRIALS} trials  (B={len(rf_vals)})')
print(f'  AUC mean : {mu:.4f}')
print(f'  AUC std  : {std:.4f}')
print(f'  90% CI   : [{p5:.3f}, {p95:.3f}]')
print(f'  Avg n    : {np.mean(nfeats):.1f}')
print(f'  vs baseline: {mu - np.mean(baseline_vals):+.4f}')
print()
print('Comparison:')
print(f'  nb21 single-run      : {NB21_RF_AUC:.4f}  delta={NB21_RF_AUC - mu:+.4f}')
print(f'  nb29 fixed-param mean: {NB29_RF_MEAN:.4f}  delta={NB29_RF_MEAN - mu:+.4f}')
print(f'  nb26 Optuna-10 mean  : {NB26_RF_MEAN:.4f}  delta={NB26_RF_MEAN - mu:+.4f}')

if mu > NB29_RF_MEAN + 0.01:
    print('\n→ Tuning helps: Optuna-30 pipeline clearly beats fixed-param baseline.')
elif mu < NB29_RF_MEAN - 0.01:
    print('\n→ Fixed params win: pipeline tuning is not adding value for RF.')
else:
    print('\n→ Tuning adds little: Optuna-30 and fixed-param means are within 0.01.')